# 검색 결과 비교 실습

# PDF DB와 Markdown DB 불러오기

## 벡터db 불러오기

In [1]:
from pathlib import Path
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

# Embedding + Chroma 저장
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
    )

# 01번 노트북에서 생성한 PDF 기반 DB
PDF_DB_PATH = Path("chroma_db/ai_trend_report_pdf")
PDF_COLLECTION_NAME = "ai_trend_report_2026_pdf"

# 02번 노트북에서 생성한 Markdown 기반 DB
MD_DB_PATH = Path("chroma_db/ai_trend_report_md")
MD_COLLECTION_NAME = "ai_trend_report_2026_md"

# PDF 기반 DB 확인
if not PDF_DB_PATH.exists():
    raise FileNotFoundError(
        f"PDF 기반 ChromaDB가 없습니다: {PDF_DB_PATH}\n"
        "먼저 01_naive_rag_limits.ipynb를 실행해 PDF 기반 DB를 생성하세요."
    )

# Markdown 기반 DB 확인
if not MD_DB_PATH.exists():
    raise FileNotFoundError(
        f"Markdown 기반 ChromaDB가 없습니다: {MD_DB_PATH}\n"
        "먼저 현재 노트북에서 Markdown 기반 DB를 생성하세요."
    )

# PDF 기반 Vector Store
pdf_store = Chroma(
    embedding_function=embeddings,
    collection_name=PDF_COLLECTION_NAME,
    persist_directory=str(PDF_DB_PATH)
)

# Markdown 기반 Vector Store
md_store = Chroma(
    embedding_function=embeddings,
    collection_name=MD_COLLECTION_NAME,
    persist_directory=str(MD_DB_PATH)
)

print("PDF 기반 DB 로드 완료")
print(f"PDF 기반 DB 청크 수: {pdf_store._collection.count()}")

print("\nMarkdown 기반 DB 로드 완료")
print(f"Markdown 기반 DB 청크 수: {md_store._collection.count()}")

PDF 기반 DB 로드 완료
PDF 기반 DB 청크 수: 52

Markdown 기반 DB 로드 완료
Markdown 기반 DB 청크 수: 29


## 검색 결과 출력 함수

In [2]:
def print_retrieval_results(store, query: str, k: int = 3, title: str = "검색 결과"):
    """
    하나의 Vector Store에 대해 검색 결과를 출력한다.
    
    store: Chroma Vector Store
    query: 검색 질문
    k: 검색할 문서 수
    title: 출력 제목
    """

    results = store.similarity_search_with_score(query, k=k)

    print("=" * 100)
    print(f"[{title}]")
    print(f"질문: {query}")
    print(f"검색 개수 k: {k}")
    print("score는 거리값에 가까우며, 낮을수록 더 유사함")
    print("=" * 100)

    for i, (doc, score) in enumerate(results, start=1):
        metadata = doc.metadata

        source = metadata.get("source", "?")
        page = metadata.get("page", None)
        page_label = page + 1 if isinstance(page, int) else "?"
        chunk_id = metadata.get("chunk_id", "?")
        doc_type = metadata.get("doc_type", "pdf_or_unknown")

        content = doc.page_content[:600].replace("\n", " ")

        print(f"\n[{i}] score={score:.4f}")
        print(f"source   : {source}")
        print(f"page     : {page_label}")
        print(f"chunk_id : {chunk_id}")
        print(f"doc_type : {doc_type}")
        print(f"content  : {content}")

## PDF 기반 DB와 Markdown 기반 DB 비교

In [3]:
def compare_pdf_md_retrieval(query: str, k: int = 3):
    """
    같은 질문에 대해 PDF 기반 DB와 Markdown 기반 DB의 검색 결과를 비교한다.
    """

    print("\n\n")
    print("#" * 100)
    print("[비교 질문]")
    print(query)
    print("#" * 100)

    print("\n\n")
    print_retrieval_results(
        store=pdf_store,
        query=query,
        k=k,
        title="PDF 직접 로딩 기반 검색 결과"
    )

    print("\n\n")
    print_retrieval_results(
        store=md_store,
        query=query,
        k=k,
        title="Markdown 변환 기반 검색 결과"
    )

## 비교 실행

In [4]:
compare_pdf_md_retrieval(
    query="산업·기술·정책 분야별 데이터 수집 키워드는 각각 무엇인가요?",
    k=3
)




####################################################################################################
[비교 질문]
산업·기술·정책 분야별 데이터 수집 키워드는 각각 무엇인가요?
####################################################################################################



[PDF 직접 로딩 기반 검색 결과]
질문: 산업·기술·정책 분야별 데이터 수집 키워드는 각각 무엇인가요?
검색 개수 k: 3
score는 거리값에 가까우며, 낮을수록 더 유사함

[1] score=0.7356
source   : data\AI@Data_Report_토픽_분석을_통한_AI_주요_트렌드_및_2026_전망_251223(최종).pdf
page     : 10
chunk_id : 20
doc_type : pdf_or_unknown
content  : 분야별로 설정한 7개 키워드를 기반으로 전체 수집 건수는 282건(분야별 94건)이며, 각 기사·리포트별로 주요 내용 요약본을 정리한 후, 산업·기술·정책별 세부 키워드 분석을 실시함Ÿ해외 매체는 의미 왜곡을 최소화하고 분석 기준을 일관되게 유지하기 위해 한국어로 번역 후 요약하였으며, 이를 통해 분석 절차의 체계성과 비교 가능성을 확보함 분야 사용 키워드AI산업AI투자, 시장규모, 기업도입, 생성형AI, 펀딩, 스타트업, 산업전환AI기술AI에이전트, 멀티모달, LLM, 온디바이스AI, 오픈소스, 딥시크, 모델개발AI정책AI규제, AI기본법, EU AI Act, 거버넌스, 행정명령, 데이터보호, 인프라정책 [ 그림 1 ]분야별 데이터셋 구성 예시

[2] score=0.8337
source   : data\AI@Data_Report_토픽_분석을_통한_AI_주요_트렌드_및_2026_전망_251223(최종).pdf
page     : 10
chunk_id : 19
do

In [5]:
test_queries = [
    "2026년 AI 기술 전망에서 핵심 변화는 무엇인가요?",
    "내년 AI 모델은 어떤 방향으로 발전할 것으로 보이나요?",
    "산업·기술·정책 분야별 데이터 수집 키워드는 각각 무엇인가요?",
    "AI 정책 분야에서 안전성과 책임이 왜 중요한 이슈인가요?",
    "토픽모델링 전에 제거한 전처리 용어 유형과 예시는 무엇인가요?"
]

for query in test_queries:
    compare_pdf_md_retrieval(query, k=3)




####################################################################################################
[비교 질문]
2026년 AI 기술 전망에서 핵심 변화는 무엇인가요?
####################################################################################################



[PDF 직접 로딩 기반 검색 결과]
질문: 2026년 AI 기술 전망에서 핵심 변화는 무엇인가요?
검색 개수 k: 3
score는 거리값에 가까우며, 낮을수록 더 유사함

[1] score=0.7387
source   : data\AI@Data_Report_토픽_분석을_통한_AI_주요_트렌드_및_2026_전망_251223(최종).pdf
page     : 18
chunk_id : 40
doc_type : pdf_or_unknown
content  : 토픽 분석을 통한 AI 주요 트렌드 및 2026 전망18              ※ IoT 센서와 AI 기술을 통해 물류 및 공급망 전 과정을 자동화하고 최적화4)[ 그림 5 ]주요 산업 분야별 AI 적용 사례4.32026년 AI 기술 전망 2026년 AI 기술은 지능 구조 고도화와 데이터 한계 보완을 중심으로 진화할 전망임Ÿ합성데이터·추론형 AI·멀티모달 기술이 주요 경쟁 축으로 자리 잡으면서, 학습 효율 향상·복합 정보 처리·설명 가능성 강화 등 모델 내부 구조의 질적 개선 흐름이 이어질 것으로 예상됨 기술 고도화는 입력 유형과 데이터 범위를 확장함으로써 AI의 활용 가능성을 넓히고, 복잡한 상황에서의 판단 능력을 강화하는 방향으로 진화할 것으로 전망됨Ÿ고품질 데이터 생성·멀티모달·고급 추론 기술이 결합되며 AI의 상황 이해와 문제 해결 능력이 향상되고, 서비스·산업 전반의 활용도도 확대될 전망임

[2] score=0.7616
source   : data\AI@Data_Re

- Markdown 변환이 항상 검색 순위를 개선하는 것은 아니다.
- 질문 유형, 청크 크기, 섹션 경계에 따라 PDF 기반 검색이 더 직접적인 결과를 반환할 수도 있다.

[Q] 내년 AI 모델은 어떤 방향으로 발전할 것으로 보이나요?
- 이 질문은 사람이 보기에는 자연스럽지만, 검색 관점에서는 모호하다.
- ‘내년’을 ‘2026년’으로 바꾸고, ‘AI 모델’을 ‘AI 기술 전망, 합성데이터, 추론형 AI, 멀티모달’처럼 구체화하면 검색 품질이 개선될 수 있다.

# 검색 결과 + 최종 답변 비교

In [6]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

## 검색 문서를 답변용 context로 변환

In [7]:
def format_docs_with_metadata(docs):
    """
    검색된 문서를 LLM에 넣기 좋은 형태로 변환한다.
    출처, page, chunk_id를 함께 표시한다.
    """

    formatted = []

    for i, doc in enumerate(docs, start=1):
        metadata = doc.metadata

        source = metadata.get("source", "?")
        page = metadata.get("page", None)
        page_label = page + 1 if isinstance(page, int) else "?"
        chunk_id = metadata.get("chunk_id", "?")
        doc_type = metadata.get("doc_type", "?")

        formatted.append(
            f"[문서 {i}]\n"
            f"source: {source}\n"
            f"page: {page_label}\n"
            f"chunk_id: {chunk_id}\n"
            f"doc_type: {doc_type}\n\n"
            f"{doc.page_content}"
        )

    return "\n\n" + "=" * 80 + "\n\n".join(formatted)

## RAG 답변 생성 

In [16]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

answer_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
아래 문서만 참고하여 질문에 답하세요.

규칙:
1. 문서에 있는 내용만 사용하세요.
2. 문서에 없는 내용은 추측하지 마세요.
3. 표나 목록 정보가 있으면 항목별로 정리하세요.
4. PDF 기반 답변과 Markdown 기반 답변을 비교하기 위한 실습이므로, 근거가 되는 표현을 최대한 문서에 맞춰 답하세요.

[문서]
{context}
"""
    ),
    ("human", "{question}")
])

# 예시 질문 
answer_chain = answer_prompt | llm | StrOutputParser()

# pdf 기반 검색 결과 테스트
question = "산업·기술·정책 분야별 데이터 수집 키워드는 각각 무엇인가요?"
docs = pdf_store.similarity_search(question, k=3)
context = format_docs_with_metadata(docs)

answer = answer_chain.invoke({
    "context": context, 
    "question": question
    })
print("답변:")
print(answer)

답변:
산업·기술·정책 분야별 데이터 수집 키워드는 다음과 같습니다.

### 산업 분야
- AI투자
- 시장규모
- 기업도입
- 생성형AI
- 펀딩
- 스타트업
- 산업전환

### 기술 분야
- AI에이전트
- 멀티모달
- LLM
- 온디바이스AI
- 오픈소스
- 딥시크
- 모델개발

### 정책 분야
- AI규제
- AI기본법
- EU AI Act
- 거버넌스
- 행정명령
- 데이터보호
- 인프라정책


## 하나의 VectorDB에서 검색과 답변을 함께 실행

In [10]:
def run_rag_with_store(store, question: str, k: int = 3):
    """
    하나의 Vector Store에 대해 검색 결과와 최종 답변을 함께 반환한다.
    """

    retriever = store.as_retriever(search_kwargs={"k": k})
    docs = retriever.invoke(question)

    context = format_docs_with_metadata(docs)

    answer = answer_chain.invoke({
        "context": context,
        "question": question
    })

    return docs, answer

## PDF 기반 RAG와 Markdown 기반 RAG 답변 비교

In [11]:
def compare_pdf_md_rag_answer(question: str, k: int = 3):
    """
    같은 질문에 대해 PDF 기반 RAG와 Markdown 기반 RAG의 최종 답변을 비교한다.
    """

    print("#" * 100)
    print("[비교 질문]")
    print(question)
    print("#" * 100)

    # PDF 기반 RAG
    pdf_docs, pdf_answer = run_rag_with_store(
        store=pdf_store,
        question=question,
        k=k
    )

    # Markdown 기반 RAG
    md_docs, md_answer = run_rag_with_store(
        store=md_store,
        question=question,
        k=k
    )

    print("\n\n" + "=" * 100)
    print("[PDF 기반 RAG 답변]")
    print("=" * 100)
    print(pdf_answer)

    print("\n\n" + "=" * 100)
    print("[Markdown 기반 RAG 답변]")
    print("=" * 100)
    print(md_answer)

    return {
        "question": question,
        "pdf_answer": pdf_answer,
        "md_answer": md_answer,
        "pdf_docs": pdf_docs,
        "md_docs": md_docs
    }

## 실행 예시

In [12]:
result = compare_pdf_md_rag_answer(
    question="산업·기술·정책 분야별 데이터 수집 키워드는 각각 무엇인가요?",
    k=3
)

####################################################################################################
[비교 질문]
산업·기술·정책 분야별 데이터 수집 키워드는 각각 무엇인가요?
####################################################################################################


[PDF 기반 RAG 답변]
산업·기술·정책 분야별 데이터 수집 키워드는 다음과 같습니다.

### 산업 분야
- AI투자
- 시장규모
- 기업도입
- 생성형AI
- 펀딩
- 스타트업
- 산업전환

### 기술 분야
- AI에이전트
- 멀티모달
- LLM
- 온디바이스AI
- 오픈소스
- 딥시크
- 모델개발

### 정책 분야
- AI규제
- AI기본법
- EU AI Act
- 거버넌스
- 행정명령
- 데이터보호
- 인프라정책


[Markdown 기반 RAG 답변]
산업·기술·정책 분야별 데이터 수집 키워드는 다음과 같습니다.

## **분야별 데이터 수집 키워드**

| 분야   | 사용 키워드                                      |
|--------|-------------------------------------------------|
| AI산업 | AI투자, 시장규모, 기업도입, 생성형AI, 펀딩, 스타트업, 산업전환 |
| AI기술 | AI에이전트, 멀티모달, LLM, 온디바이스AI, 오픈소스, 딥시크, 모델개발 |
| AI정책 | AI규제, AI기본법, EU AI Act, 거버넌스, 행정명령, 데이터보호, 인프라정책 |


In [ ]:
compare_pdf_md_rag_answer(
    question="토픽모델링 전에 제거한 전처리 용어 유형과 예시는 무엇인가요?",
    k=3
)

####################################################################################################
[비교 질문]
토픽모델링 전에 제거한 전처리 용어 유형과 예시는 무엇인가요?
####################################################################################################


[PDF 기반 RAG 답변]
전처리 과정에서 제거한 용어 유형과 예시는 다음과 같습니다.

1. **숫자**
   - 예시: 0~9 (분석에 불필요한 단어)

2. **1음절 단어**
   - 예시: 수, 올, 등, 것 등 (정보성 낮은 단어)

3. **불용어**
   - 예시: 시점, 구분어, 접미/접두사 (분석에 불필요한 일반어, 구분어)

4. **국가 이름**
   - 예시: 한국, 대한민국, 미국, 일본, 중국, 유럽, 세계 등 (지명, 국가명)

5. **조사**
   - 예시: 의, 는, 에, 에서, 으로, 도, 와, 과 등 (문법적 기능, 의미성 적음)

6. **기타 불용어**
   - 예시: 관련, 기반, 중심, 수준, 부분, 측면, 방식, 올해, 내년, 작년, 최근, 리포트, 보고서, 발표, 기사, 보도, 분석, 딥시크, 진행, 달러, 전체, 기술전문지, 기술전문매체, 경제지, 통신사, 매체, 예정, 분야, 모델, 공개, 투자, 산업, 기술, 정책, 논의


[Markdown 기반 RAG 답변]
전처리 과정에서 제거한 용어 유형과 예시는 다음과 같습니다.

|**분류**|**주요 예시들**|**특징**|
|---|---|---|
|숫자|0~9|분석에 불필요한 단어|
|1음절 단어|수, 올, 등, 것 등|정보성 낮은 단어|
|불용어|시점, 구분어, 접미/접두사|분석에 불필요한 일반어, 구분어|
|국가 이름|한국, 대한민국, 미국, 일본, 중국, 유럽, 세계 등|지명, 국가명|
|조사|의, 는, 에, 에서, 

{'question': '토픽모델링 전에 제거한 전처리 용어 유형과 예시는 무엇인가요?',
 'pdf_answer': '전처리 과정에서 제거한 용어 유형과 예시는 다음과 같습니다.\n\n1. **숫자**\n   - 예시: 0~9 (분석에 불필요한 단어)\n\n2. **1음절 단어**\n   - 예시: 수, 올, 등, 것 등 (정보성 낮은 단어)\n\n3. **불용어**\n   - 예시: 시점, 구분어, 접미/접두사 (분석에 불필요한 일반어, 구분어)\n\n4. **국가 이름**\n   - 예시: 한국, 대한민국, 미국, 일본, 중국, 유럽, 세계 등 (지명, 국가명)\n\n5. **조사**\n   - 예시: 의, 는, 에, 에서, 으로, 도, 와, 과 등 (문법적 기능, 의미성 적음)\n\n6. **기타 불용어**\n   - 예시: 관련, 기반, 중심, 수준, 부분, 측면, 방식, 올해, 내년, 작년, 최근, 리포트, 보고서, 발표, 기사, 보도, 분석, 딥시크, 진행, 달러, 전체, 기술전문지, 기술전문매체, 경제지, 통신사, 매체, 예정, 분야, 모델, 공개, 투자, 산업, 기술, 정책, 논의',
 'md_answer': '전처리 과정에서 제거한 용어 유형과 예시는 다음과 같습니다.\n\n|**분류**|**주요 예시들**|**특징**|\n|---|---|---|\n|숫자|0~9|분석에 불필요한 단어|\n|1음절 단어|수, 올, 등, 것 등|정보성 낮은 단어|\n|불용어|시점, 구분어, 접미/접두사|분석에 불필요한 일반어, 구분어|\n|국가 이름|한국, 대한민국, 미국, 일본, 중국, 유럽, 세계 등|지명, 국가명|\n|조사|의, 는, 에, 에서, 으로, 도, 와, 과 등|문법적 기능, 의미성 적음|\n|기타 불용어|관련, 기반, 중심, 수준, 부분, 측면, 방식, 올해, 내년, 작년, 최근, 리포트, 보고서, 발표, 기사, 보도, 분석, 딥시크, 진행, 달러, 전체, 기술전문지, 기술전문매체, 경제지, 통신사, 매체, 예정, 분야, 모델, 공개, 투자, 산

### 비교 결과 확인 포인트

PDF 직접 로딩 기반 DB와 Markdown 기반 DB를 같은 질문으로 비교하면 다음 차이를 확인할 수 있다.

1. PDF 기반 검색은 페이지 단위 metadata가 남아 있어 page 정보를 확인하기 쉽다.
2. Markdown 기반 검색은 page 정보는 없지만, 제목·문단·표 구조가 더 잘 보존될 수 있다.
3. 표나 절차 정보는 Markdown 기반 검색에서 더 자연스럽게 검색될 가능성이 있다.
4. 다만 Markdown도 chunk 크기와 분할 방식에 따라 행·열 관계가 끊길 수 있다.
5. 따라서 PDF → Markdown 변환은 검색 품질 개선에 도움이 되지만, 그 자체로 모든 RAG 문제가 해결되지는 않는다.